In [1]:
import pandas as pd
import altair as alt
import numpy as np

import sys
sys.path.insert(0, '..')

from dataloader import load_df, load_perframe_pervalue, PARAM_LABELS, PARAMS,RESOLUTIONS, RESOLUTION_LABELS

JSON_PATH = "../dataset.json"  

In [7]:

# ── Load data ────────────────────────────────────────────────────────────────

df = load_df(JSON_PATH, center=True)
df1 = load_df(JSON_PATH, center=False)
df_pf = load_perframe_pervalue(JSON_PATH)


/Users/mariasilva/Documents/PerceptualTAA/parameter_analysis/../dataloader.py:103: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(_center_group)


#### Per-Scene Quality Range After Baseline Centering (Mean ± SD Across Scenes)
- For each `(scene, resolution, parameter)` group,  takes all the scores at different parameter values and does max - min.
- So for example, (canyon, 100%, num_samples) might have scores [82, 84, 87, 85] across four num_samples settings — the range is 87 - 82 = 5.
- 


In [24]:
# 2. Update the aggregation to use 'score_centered'
range_per_scene = (
    df.groupby(['scene', 'resolution', 'parameter'])['score'] # <-- Use centered here
    .agg(lambda x: x.max() - x.min())
    .reset_index()
    .rename(columns={'score': 'score_range'})
)

# 3. The rest of your table logic remains the same
def mean_std(x):
    return f"{x.mean():.2f} ± {x.std():.2f}"

table1 = (
    range_per_scene
    .groupby(['resolution', 'parameter'])['score']
    .agg(mean_std)
    .unstack('parameter')
    .rename(columns=PARAM_LABELS)
    .sort_index(ascending=False)
    .rename(index=RESOLUTION_LABELS)
)

table1

KeyError: 'Column not found: score'

In [9]:
print(df['score_range'])

KeyError: 'score_range'

In [ ]:
print(df['score'])

0       95.590088
1       95.765892
2       96.069374
3       96.245041
4       96.611214
          ...    
2235    77.789734
2236    78.703194
2237    77.977646
2238    78.675255
2239    77.890625
Name: score, Length: 2240, dtype: float64


In [ ]:
UNREAL_DEFAULT_ALPHA = 0.04

alpha_df = df[df['parameter'] == 'alpha_weight'].copy()

mean_scores = (
    alpha_df.groupby(['resolution', 'value'])['score']
    .mean()
    .reset_index()
)

# Optimal (argmax) per resolution
optimal = mean_scores.loc[mean_scores.groupby('resolution')['score'].idxmax()].copy()
optimal = optimal.rename(columns={'value': 'optimal_alpha', 'score': 'optimal_score'})

# Baseline score at Unreal default
baseline = (
    mean_scores[mean_scores['value'] == UNREAL_DEFAULT_ALPHA]
    .rename(columns={'score': 'baseline_score'})
    [['resolution', 'baseline_score']]
)

# Join and compute gain
result = optimal.merge(baseline, on='resolution')
result['gain'] = result['optimal_score'] - result['baseline_score']
result['resolution'] = result['resolution'].map(RESOLUTION_LABELS)
result = result.sort_values('resolution', ascending=False)

display(result[['resolution', 'baseline_score', 'optimal_alpha', 'optimal_score', 'gain']].round(2))

,resolution,baseline_score,optimal_alpha,optimal_score,gain
2,75%,92.39,0.5,94.28,1.89
1,50%,91.14,0.2,92.64,1.50
0,25%,87.35,0.2,88.60,1.25
3,100%,93.24,0.6,95.26,2.02


In [ ]:
# 1. Filter for Alpha and separate Unreal Default (Baseline)
alpha_df = df[df['parameter'] == 'alpha_weight'].copy()
baseline_val = 0.04

# 2. Get baseline score per scene/resolution
baselines = (
    alpha_df[alpha_df['value'] == baseline_val]
    .set_index(['scene', 'resolution'])['score']
    .rename('baseline_score')
)

# 3. Get the maximum possible score per scene/resolution (The "Ideal" tuning)
per_scene_opt = (
    alpha_df.groupby(['scene', 'resolution'])['score']
    .max()
    .rename('optimal_score')
)

# 4. Combine and calculate the gain per scene
scene_gains = pd.merge(per_scene_opt, baselines, left_index=True, right_index=True).reset_index()
scene_gains['gain'] = scene_gains['optimal_score'] - scene_gains['baseline_score']

# 5. Aggregate: Find Average, Best (Max), and Worst (Min) Gain per Resolution
gain_summary = (
    scene_gains.groupby('resolution')['gain']
    .agg([
        ('Avg Gain', 'mean'),
        ('Best Case (Max)', 'max'),
        ('Worst Case (Min)', 'min'),
        ('Std Dev', 'std')
    ])
    .sort_index(ascending=False)
)

# Map labels for readability
gain_summary.index = gain_summary.index.map(RESOLUTION_LABELS)
display(gain_summary.round(2))

,Avg Gain,Best Case (Max),Worst Case (Min),Std Dev
resolution,,,,
100%,2.11,7.54,0.14,2.29
75%,1.94,6.54,0.13,1.96
50%,1.67,4.66,0.12,1.46
25%,1.27,3.96,0.13,0.97


In [ ]:
# 1. Filter for Alpha and separate Unreal Default (Baseline)
alpha_df = df[df['parameter'] == 'hist_percent'].copy()
baseline_val = 100

# 2. Get baseline score per scene/resolution
baselines = (
    alpha_df[alpha_df['value'] == baseline_val]
    .set_index(['scene', 'resolution'])['score']
    .rename('baseline_score')
)

# 3. Get the maximum possible score per scene/resolution (The "Ideal" tuning)
per_scene_opt = (
    alpha_df.groupby(['scene', 'resolution'])['score']
    .max()
    .rename('optimal_score')
)

# 4. Combine and calculate the gain per scene
scene_gains = pd.merge(per_scene_opt, baselines, left_index=True, right_index=True).reset_index()
scene_gains['gain'] = scene_gains['optimal_score'] - scene_gains['baseline_score']

# 5. Aggregate: Find Average, Best (Max), and Worst (Min) Gain per Resolution
gain_summary = (
    scene_gains.groupby('resolution')['gain']
    .agg([
        ('Avg Gain', 'mean'),
        ('Best Case (Max)', 'max'),
        ('Worst Case (Min)', 'min'),
        ('Std Dev', 'std')
    ])
    .sort_index(ascending=False)
)

# Identify which specific hist_percent value provided the 'Best Case'
best_values = (
    df[df['parameter'] == 'hist_percent']
    .sort_values('score', ascending=False)
    .drop_duplicates(['scene', 'resolution'])
    [['scene', 'resolution', 'value', 'score']]
)

# Merge back with your scene_gains to see the 'Best Value' per scene
scene_details = pd.merge(scene_gains, best_values, on=['scene', 'resolution'])

# Summary of which values are "Winning" most often
value_counts = scene_details.groupby(['resolution', 'value']).size().unstack(fill_value=0)
value_counts.index = value_counts.index.map(RESOLUTION_LABELS)
print("Frequency of each History % being the 'Optimal' choice:")
display(value_counts)

# Map labels for readability
gain_summary.index = gain_summary.index.map(RESOLUTION_LABELS)
display(gain_summary.round(2))

Frequency of each History % being the 'Optimal' choice:


value,100.0,125.0,150.0,200.0
resolution,,,,
25%,2,4,11,3
50%,0,1,2,17
75%,0,0,3,17
100%,0,0,3,17


,Avg Gain,Best Case (Max),Worst Case (Min),Std Dev
resolution,,,,
100%,1.37,4.16,0.07,1.19
75%,1.43,4.31,0.09,1.27
50%,1.31,4.31,0.08,1.21
25%,0.68,3.31,0.00,0.97


In [ ]:
# 1. Filter for History %
hist_df = df[df['parameter'] == 'hist_percent'].copy()
baseline_val = 100

# 2. Map baseline scores to every row for easy subtraction
baselines = (
    hist_df[hist_df['value'] == baseline_val]
    .groupby(['scene', 'resolution'])['score']
    .mean() # Should only be one value per scene/res
)

hist_df = hist_df.merge(baselines, on=['scene', 'resolution'], suffixes=('', '_baseline'))
hist_df['delta_from_baseline'] = hist_df['score'] - hist_df['score_baseline']

# 3. Aggregate results
# "Worst Possible" is the MIN delta (can be negative)
# "Best Possible" is the MAX delta (your previous 'gain')
summary_with_negatives = (
    hist_df.groupby('resolution')['delta_from_baseline']
    .agg([
        ('Avg Potential Change', 'mean'),
        ('Max Improvement (Best)', 'max'),
        ('Max Degradation (Worst)', 'min'), 
        ('Std Dev', 'std')
    ])
    .sort_index(ascending=False)
)

summary_with_negatives.index = summary_with_negatives.index.map(RESOLUTION_LABELS)
display(summary_with_negatives.round(2))

,Avg Potential Change,Max Improvement (Best),Max Degradation (Worst),Std Dev
resolution,,,,
100%,0.71,4.16,-0.00,0.91
75%,0.76,4.31,0.00,0.95
50%,0.69,4.31,-1.11,0.92
25%,0.08,3.31,-3.04,1.00


In [10]:
import pandas as pd

# 1. Setup
PARAM = 'hist_percent'
BASELINE_VAL = 100.0
alpha_df = df[df['parameter'] == PARAM].copy()

# 2. Get baseline score per scene/resolution
baselines = (
    alpha_df[alpha_df['value'] == BASELINE_VAL]
    .groupby(['scene', 'resolution'])['score']
    .mean()
    .reset_index()
    .rename(columns={'score': 'baseline_score'})
)

# 3. Merge baseline back to calculate all deltas
# This allows us to see how EVERY value performed relative to the 100% default
all_data = alpha_df.merge(baselines, on=['scene', 'resolution'])
all_data['delta'] = all_data['score'] - all_data['baseline_score']

# 4. Calculate Per-Scene Optimal Gain (The "Upside")
# For every scene, what is the best possible improvement we could get?
per_scene_opt = (
    all_data.groupby(['scene', 'resolution'])['delta']
    .max() # The best gain possible for this specific scene
    .reset_index()
    .rename(columns={'delta': 'optimal_gain'})
)

# 5. Aggregate everything by Resolution
# Average Gain: Average of the best possible gains per scene
# Best Case: The single highest gain in the whole dataset
# Max Degradation: The single lowest delta in the whole dataset (The Risk)
summary = (
    per_scene_opt.groupby('resolution')['optimal_gain']
    .agg([
        ('Avg Optimal Gain', 'mean'),
        ('Best Case (Max)', 'max'),
        ('Std Dev', 'std')
    ])
)

# Add the "Worst Case Degradation" from the full dataset (all_data)
# This looks at every value, not just the 'optimal' ones.
summary['Max Degradation (Risk)'] = all_data.groupby('resolution')['delta'].min()

# 6. Find the "Culprit" value for the Max Degradation
worst_idx = all_data.groupby('resolution')['delta'].idxmin()
worst_values = all_data.loc[worst_idx, ['resolution', 'value']].set_index('resolution')
summary['At Value (Worst)'] = worst_values['value']

# 7. Final Formatting
summary.index = summary.index.map(RESOLUTION_LABELS)
summary = summary.sort_index(ascending=False)

print(f"--- Unified Analysis for {PARAM} ---")
display(summary.round(2))

--- Unified Analysis for hist_percent ---


,Avg Optimal Gain,Best Case (Max),Std Dev,Max Degradation (Risk),At Value (Worst)
resolution,,,,,
75%,1.43,4.31,1.27,0.00,100.0
50%,1.31,4.31,1.21,-1.11,200.0
25%,0.68,3.31,0.97,-3.04,200.0
100%,1.37,4.16,1.19,-0.00,125.0


In [11]:
import matplotlib.pyplot as plt

# Prepare data for plotting
plot_data = summary_with_negatives.sort_index() # Sort 25% to 100%

plt.figure(figsize=(10, 6))
res = plot_data.index
avg = plot_data['Avg Potential Change']
hi  = plot_data['Max Improvement (Best)']
lo  = plot_data['Max Degradation (Worst)']

# Plot the average line
plt.plot(res, avg, marker='o', label='Average Change', color='blue', linewidth=2)

# Fill the area between Best and Worst (The "Envelope")
plt.fill_between(res, lo, hi, color='blue', alpha=0.1, label='Range of Outcomes')

# Add a "Zero Line" - anything below this made the quality worse!
plt.axhline(0, color='red', linestyle='--', alpha=0.6, label='Unreal Default Baseline')

plt.title('Impact of History % Buffer Size on CGVQM Score')
plt.ylabel('CGVQM Point Delta (vs. 100% History)')
plt.xlabel('Resolution')
plt.grid(axis='y', alpha=0.3)
plt.legend()
plt.savefig('history_impact_envelope.png')

NameError: name 'summary_with_negatives' is not defined

In [13]:
UNREAL_DEFAULT_HIST = 100

alpha_df = df[df['parameter'] == 'hist_percent'].copy()

mean_scores = (
    alpha_df.groupby(['resolution', 'value'])['score']
    .mean()
    .reset_index()
)

# Optimal (argmax) per resolution
optimal = mean_scores.loc[mean_scores.groupby('resolution')['score'].idxmax()].copy()
optimal = optimal.rename(columns={'value': 'optimal_hist', 'score': 'optimal_score'})

# Baseline score at Unreal default
baseline = (
    mean_scores[mean_scores['value'] == UNREAL_DEFAULT_HIST]
    .rename(columns={'score': 'baseline_score'})
    [['resolution', 'baseline_score']]
)

# Join and compute gain
result = optimal.merge(baseline, on='resolution')
result['gain'] = result['optimal_score'] - result['baseline_score']
result['resolution'] = result['resolution'].map(RESOLUTION_LABELS)
result = result.sort_values('resolution', ascending=False)

display(result[['resolution', 'baseline_score', 'optimal_hist', 'optimal_score', 'gain']].round(2))

,resolution,baseline_score,optimal_hist,optimal_score,gain
2,75%,92.41,200.0,93.82,1.41
1,50%,91.13,200.0,92.35,1.22
0,25%,87.35,150.0,87.74,0.38
3,100%,93.34,200.0,94.70,1.36


In [14]:
res_order  = ['100%', '87%', '71%', '50%']
res_colors = ['#2a7db5', '#2ca05a', '#e07b3d', '#d4504a']
color_scale = alt.Scale(domain=res_order, range=res_colors)

In [15]:
def build_chart(param_name, title):
    param_df = df[df['parameter'] == param_name].copy()
    
    mean_scores = (
        param_df.groupby(['resolution', 'value'])['score']
        .mean().reset_index()
    )
    mean_scores['resolution_label'] = mean_scores['resolution'].map(RESOLUTION_LABELS)
    
    scene_scores = (
        param_df.groupby(['scene', 'resolution', 'value'])['score']
        .mean().reset_index()
    )
    scene_scores['resolution_label'] = scene_scores['resolution'].map(RESOLUTION_LABELS)
    
    optimal_vals = (
        mean_scores.loc[mean_scores.groupby('resolution')['score'].idxmax(), ['resolution','value']]
        .rename(columns={'value': 'opt_val'})
    )
    mean_scores = mean_scores.merge(optimal_vals, on='resolution', how='left')
    mean_scores['is_optimal'] = mean_scores['value'] == mean_scores['opt_val']
    
    DEFAULTS = {'alpha_weight': 0.04, 'hist_percent': 100}
    default_val = DEFAULTS[param_name]
    mean_scores['is_default'] = mean_scores['value'] == default_val

    # Global y domain across all scenes and resolutions
    y_min = scene_scores['score'].min() - 1
    y_max = scene_scores['score'].max() + 1
    y_scale = alt.Scale(domain=[y_min, y_max], zero=False)

    panels = []
    for res_label, res_int in zip(res_order, [100, 87, 71, 50]):
        ms  = mean_scores[mean_scores['resolution_label'] == res_label]
        ss  = scene_scores[scene_scores['resolution_label'] == res_label]
        opt = ms[ms['is_optimal']]
        dft = ms[ms['is_default']]

        opt_score = opt['score'].values[0]
        dft_score = dft['score'].values[0]
        gain_pct  = (opt_score - dft_score) / dft_score * 100
        gain_sign = '+' if gain_pct >= 0 else ''

        annotation_df = opt.copy()
        annotation_df['label'] = f'Δ = {gain_sign}{gain_pct:.1f}%'

        scene_l = alt.Chart(ss).mark_line(opacity=0.3, strokeWidth=0.8).encode(
            x=alt.X('value:Q', title=PARAM_LABELS[param_name]),
            y=alt.Y('score:Q',
                    title='CGVQM' if res_label == '100%' else '',
                    scale=y_scale),
            color=alt.Color('resolution_label:N', scale=color_scale, legend=None),
            detail='scene:N',
        )
        mean_l = alt.Chart(ms).mark_line(strokeWidth=2.5).encode(
            x='value:Q',
            y=alt.Y('score:Q', scale=y_scale),
            color=alt.Color('resolution_label:N', scale=color_scale, legend=None),
        )
        dft_rule = alt.Chart(dft).mark_rule(
            strokeDash=[4, 2], strokeWidth=2, color='#f5c400'
        ).encode(x='value:Q')
        dft_dot = alt.Chart(dft).mark_point(
            size=80, filled=True, color='#f5c400', shape='diamond'
        ).encode(
            x='value:Q',
            y=alt.Y('score:Q', scale=y_scale),
        )
        opt_rule = alt.Chart(opt).mark_rule(
            strokeDash=[4, 2], strokeWidth=2, color='#2ca05a'
        ).encode(x='value:Q')
        opt_dot = alt.Chart(opt).mark_point(
            size=80, filled=True, color='#2ca05a'
        ).encode(
            x='value:Q',
            y=alt.Y('score:Q', scale=y_scale),
        )
        annotation = alt.Chart(annotation_df).mark_text(
            align='left', dx=6, dy=-10,
            fontSize=10, fontWeight='bold', color='#2ca05a'
        ).encode(
            x='value:Q',
            y=alt.Y('score:Q', scale=y_scale),
            text='label:N',
        )

        panel = (
            scene_l + mean_l + dft_rule + dft_dot + opt_rule + opt_dot + annotation
        ).properties(width=180, height=180, title=res_label)
        panels.append(panel)

    return alt.hconcat(*panels).properties(title=title)


alpha_chart = build_chart('alpha_weight', 'Alpha weight vs. mean CGVQM score by resolution')
hist_chart  = build_chart('hist_percent',  'History % vs. mean CGVQM score by resolution')

alt.vconcat(alpha_chart, hist_chart)
# alpha_chart

IndexError: index 0 is out of bounds for axis 0 with size 0

In [16]:
UNREAL_DEFAULTS = {
        'alpha_weight': 0.04,
        'hist_percent': 100,
        'filter_size': 1,
        'num_samples': 8
    }

def plot_response_curves(input_df, score_col, y_title, plot_title, y_min=None, y_max=None):
    display_map = {'100%': '100%', '87%': '75%', '71%': '50%', '50%': '25%'}

    agg = (
        input_df.groupby(['parameter', 'resolution', 'value'])[score_col]
        .agg(['mean', 'std', 'count'])
        .reset_index()
    )
    agg['ci95'] = 1.96 * (agg['std'] / np.sqrt(agg['count']))
    agg['resolution'] = agg['resolution'].astype(str) + '%'
    agg['res_label'] = agg['resolution'].map(display_map)
    
    agg['ci_low']  = agg['mean'] - agg['std']
    agg['ci_high'] = agg['mean'] + agg['std']

    _y_min = y_min if y_min is not None else agg['ci_low'].min()  - 0.5
    _y_max = y_max if y_max is not None else agg['ci_high'].max() + 0.5
    y_scale = alt.Scale(domain=[_y_min, _y_max])

    charts = []
    for param in PARAMS:
        sub = agg[agg['parameter'] == param]
        default_val = UNREAL_DEFAULTS.get(param)        
        # 1. Define the shared color encoding once
        # The mapping logic for the labels
        label_map = "datum.value == '100%' ? '100%' : datum.value == '87%' ? '75%' : datum.value == '71%' ? '50%' : '25%'"

        color_encoding = alt.Color(
            'resolution:N', 
            title='Base Resolution', 
            # Keep your scale here to preserve the HEX colors
            scale=alt.Scale(
                domain=res_order, # ['100%', '87%', '71%', '50%']
                range=res_colors  # ['#2a7db5', '#2ca05a', '#e07b3d', '#d4504a']
            ),
            sort=res_order,
            # The legend only overrides the TEXT, not the color mapping
            legend=alt.Legend(
                labelExpr=label_map,
                orient='right'
            )
        )

        # 2. Base chart holds the data and the shared X/Color encodings
        base = alt.Chart(sub).encode(
            x=alt.X('value:Q', title=PARAM_LABELS[param]),
            color=color_encoding
        )

        band = base.mark_area(opacity=0.2).encode(
            y=alt.Y('ci_low:Q', title='', scale=y_scale),
            y2=alt.Y2('ci_high:Q')
        )

        line = base.mark_line(point=True).encode(
            y=alt.Y('mean:Q', title=y_title, scale=y_scale),
            tooltip=[
                'resolution', 'value',
                alt.Tooltip('mean:Q', format='.3f'),
                alt.Tooltip('std:Q', format='.3f')
            ]
        )

        layers = [band, line]
        if default_val is not None:
            unreal_line = alt.Chart(pd.DataFrame({'x': [default_val]})).mark_rule(
                color='#FFD700', 
                strokeDash=[4, 4], 
                size=2
            ).encode(
                x='x:Q'
            )
            layers.append(unreal_line)

        # 4. Layer them all together
        charts.append(
            alt.layer(*layers).properties(
                title=PARAM_LABELS[param], 
                width=250, 
                height=220
            )
        )

    return (
        alt.hconcat(*charts)
        .resolve_scale(color='shared')
        .properties(title=plot_title)
    )


# Raw scores
df_raw = load_df(JSON_PATH, center=False)

# pre-compute bounds so we can pass tight y limits
agg_raw = (
    df_raw.groupby(['parameter', 'resolution', 'value'])['score']
    .agg(['mean', 'std'])
    .reset_index()
)
raw_y_min = (agg_raw['mean'] - agg_raw['std']).min() - 0.5
raw_y_max = (agg_raw['mean'] + agg_raw['std']).max() + 0.5

raw_chart = plot_response_curves(
    df_raw,
    score_col='score',
    y_title='Mean CGVQM Score',
    plot_title='TAA Parameter Response Curves (Raw CGVQM)',
    y_min=raw_y_min,
    y_max=raw_y_max
)

# Centered scores
df_cent = load_df(JSON_PATH, center=True)

agg_cent = (
    df_cent.groupby(['parameter', 'resolution', 'value'])['score_centered']
    .agg(['mean', 'std'])
    .reset_index()
)
cent_y_min = (agg_cent['mean'] - agg_cent['std']).min() - 0.5
cent_y_max = (agg_cent['mean'] + agg_cent['std']).max() + 0.5

cent_chart = plot_response_curves(
    df_cent,
    score_col='score_centered',
    y_title='Mean Centered CGVQM',
    plot_title='TAA Parameter Response Curves (Mean-Centered CGVQM)',
    y_min=cent_y_min,
    y_max=cent_y_max
)

alt.vconcat(raw_chart, cent_chart).resolve_scale(color='shared')

/Users/mariasilva/Documents/PerceptualTAA/parameter_analysis/../dataloader.py:103: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(_center_group)


alt.VConcatChart(...)

In [19]:
df_raw = load_df(JSON_PATH, center=False)

def compute_recommendations(df, resolution_labels=RESOLUTION_LABELS):
    recommendations = {}
    results = {}
    
    for param, default_val in UNREAL_DEFAULTS.items():
        param_df = df[df['parameter'] == param].copy()
        mean_scores = (
            param_df.groupby(['resolution', 'value'])['score']
            .mean()
            .reset_index()
        )
        # Optimal (argmax) per resolution
        optimal = mean_scores.loc[mean_scores.groupby('resolution')['score'].idxmax()].copy()
        optimal = optimal.rename(columns={'value': 'optimal_value', 'score': 'optimal_score'})
        # Baseline score at Unreal default
        baseline = (
            mean_scores[mean_scores['value'] == default_val]
            .rename(columns={'score': 'baseline_score'})
            [['resolution', 'baseline_score']]
        )
        # Join and compute gain
        result = optimal.merge(baseline, on='resolution')
        result['gain'] = result['optimal_score'] - result['baseline_score']
        result['resolution'] = result['resolution'].map(resolution_labels)
        result = result.sort_values('resolution', ascending=False)
        
        results[param] = result[['resolution', 'baseline_score', 'optimal_value', 'optimal_score', 'gain']].round(2)
        
        # {resolution_label: optimal_value} for the plot
        recommendations[param] = dict(zip(result['resolution'], result['optimal_value']))
    
    return recommendations, results

RECOMMENDATIONS, rec_results = compute_recommendations(df_raw)

def plot_response_curves(input_df, score_col, y_title, plot_title, y_min=None, y_max=None):
    display_map = {'100%': '100%', '87%': '75%', '71%': '50%', '50%': '25%'}
    reverse_map = {v: k for k, v in display_map.items()}

    # 1. Prepare Aggregated Data
    agg = (
        input_df.groupby(['parameter', 'resolution', 'value'])[score_col]
        .agg(['mean', 'std', 'count'])
        .reset_index()
    )
    # Ensure resolution is string + %
    agg['resolution'] = agg['resolution'].astype(str).str.replace('%', '') + '%'
    agg['ci_low']  = agg['mean'] - agg['std']
    agg['ci_high'] = agg['mean'] + agg['std']

    # 2. Global Y-Scale (Shared across all 4 internal plots)
    _y_min = y_min if y_min is not None else agg['ci_low'].min() - 0.5
    _y_max = y_max if y_max is not None else agg['ci_high'].max() + 0.5
    y_scale = alt.Scale(domain=[_y_min, _y_max], zero=False)

    charts = []
    # 3. Loop through all 4 Parameters (Ensure PARAMS contains all 4)
    for i, param in enumerate(PARAMS):
        sub = agg[agg['parameter'] == param]
        ue_default_val = UNREAL_DEFAULTS.get(param)        
        
        color_encoding = alt.Color(
            'resolution:N', 
            scale=alt.Scale(domain=res_order, range=res_colors),
            sort=res_order,
            legend=alt.Legend(labelExpr="datum.value == '100%' ? '100%' : datum.value == '87%' ? '75%' : datum.value == '71%' ? '50%' : '25%'")
        )

        # --- FIX: Only show Y-axis title for the first chart (Alpha Weight) ---
        current_y_title = y_title if i == 0 else None

        # Base Chart
        base = alt.Chart(sub).encode(
            x=alt.X('value:Q', title=PARAM_LABELS.get(param, param)),
            color=color_encoding
        )
        
        # Standard Deviation Band
        band = base.mark_area(opacity=0.2, clip=True).encode(
            y=alt.Y('ci_low:Q', title=current_y_title, scale=y_scale),
            y2=alt.Y2('ci_high:Q')
        )
        
        # Mean Line
        line = base.mark_line(point=True, clip=True).encode(
            y=alt.Y('mean:Q', scale=y_scale, title=current_y_title)
        )
        
        layers = [band, line]

        # 4. Side Annotations (Delta Labels)
        param_recs = RECOMMENDATIONS.get(param, {})
        rec_data = []
        for res_lab, val in param_recs.items():
            rec_data.append({'resolution': reverse_map.get(res_lab, res_lab), 'val': val})

        side_label_data = []
        for res in sub['resolution'].unique():
            res_sub = sub[sub['resolution'] == res]
            opt_val = next((item['val'] for item in rec_data if item['resolution'] == res), None)
            
            if opt_val is not None and ue_default_val is not None:
                # Find closest mean values for Delta
                score_opt = res_sub.iloc[(res_sub['value'] - opt_val).abs().argsort()[:1]]['mean'].values[0]
                score_ue  = res_sub.iloc[(res_sub['value'] - ue_default_val).abs().argsort()[:1]]['mean'].values[0]
                delta = score_opt - score_ue
                
                side_label_data.append({
                    'resolution': res, 'x': res_sub['value'].max(), 'y': res_sub.iloc[-1]['mean'],
                    'label': f"Δ={'+' if delta >= 0 else ''}{delta:.2f}"
                })

        if side_label_data:
            side_df = pd.DataFrame(side_label_data)
            res_rank = {'100%': 0, '87%': 1, '71%': 2, '50%': 3}
            side_df['rank'] = side_df['resolution'].map(res_rank)
            side_df = side_df.sort_values('rank')
            mid = (len(side_df) - 1) / 2
            
            for _, row in side_df.iterrows():
                dy_val = int((row['rank'] - mid) * 14)
                layers.append(alt.Chart(pd.DataFrame([row])).mark_text(
                    align='left', dx=8, dy=dy_val, fontSize=11, fontWeight='bold'
                ).encode(x='x:Q', y='y:Q', text='label:N', color=color_encoding))

        # 5. Unreal Default Rule (Gold)
        if ue_default_val is not None:
            layers.append(alt.Chart(pd.DataFrame({'x': [ue_default_val]})).mark_rule(
                color='#FFD700', strokeDash=[4, 4], size=1.5, clip=True
            ).encode(x='x:Q'))

        # Add panel to list
        charts.append(alt.layer(*layers).properties(width=230, height=200))

    # 6. Combine all 4 parameters into one row
    combined = alt.hconcat(*charts).resolve_scale(color='shared', y='shared').properties(title=plot_title)

    return combined.configure_axis(
        labelFontSize=12, titleFontSize=13, grid=True
    ).configure_legend(
        labelFontSize=12, titleFontSize=13, symbolOpacity=1
    ).configure_title(
        fontSize=16, anchor='middle'
    ).configure_view(stroke=None)
    # return alt.hconcat(*charts).resolve_scale(color='shared').properties(title=plot_title)

# 1. Generate the Raw CGVQM chart with Ideal lines
raw_chart = plot_response_curves(
    df_raw,
    score_col='score',
    y_title='Mean CGVQM Score',
    plot_title='TAA Parameter Response Curves (Raw CGVQM)',
    y_min=raw_y_min,
    y_max=raw_y_max
)

# 2. Generate the Mean-Centered chart with Ideal lines
cent_chart = plot_response_curves(
    df_cent,
    score_col='score_centered',
    y_title='Mean Centered CGVQM',
    plot_title='TAA Parameter Response Curves (Mean-Centered CGVQM)',
    y_min=cent_y_min,
    y_max=cent_y_max
)

# 3. Combine and display
# alt.vconcat(raw_chart, cent_chart).resolve_scale(color='shared')
cent_chart.save('centered-scores.png', scale_factor=2.0)
raw_chart.save('raw-scores.png', scale_factor=2.0)

In [22]:
def plot_param_per_scene(input_df, param_name, score_col, y_title, plot_title, y_min=None, y_max=None):
    display_map = {'100%': '100%', '87%': '75%', '71%': '50%', '50%': '25%'}
    # param_name is now a variable (e.g., 'alpha_weight' or 'hist_weight')
    param = param_name 

    # 1. Data Cleaning
    scene_data = input_df[input_df['parameter'] == param].copy()
    scene_data['resolution'] = scene_data['resolution'].astype(str).str.replace('%%', '%')
    if not scene_data['resolution'].iloc[0].endswith('%'):
        scene_data['resolution'] = scene_data['resolution'] + '%'

    mean_agg = scene_data.groupby(['resolution', 'value'])[score_col].mean().reset_index(name='mean')

    _y_min = y_min if y_min is not None else scene_data[score_col].min() - 1
    _y_max = y_max if y_max is not None else scene_data[score_col].max() + 1
    y_scale = alt.Scale(domain=[_y_min, _y_max], zero=False)

    charts = []
    summary_stats = {} 

    for res_raw in ['100%', '87%', '71%', '50%']:
        res_label = display_map[res_raw]
        res_color = res_colors[res_order.index(res_raw)]
        # res_color = res_colors[res_order.index(res_label)]
        
        sub_scenes = scene_data[scene_data['resolution'] == res_raw]
        sub_mean = mean_agg[mean_agg['resolution'] == res_raw]
        
        if sub_scenes.empty: continue

        ue_default_val = UNREAL_DEFAULTS.get(param)
        opt_val = RECOMMENDATIONS.get(param, {}).get(res_label)

        x_axis_title = f"{PARAM_LABELS[param]}, Resolution {res_label}"

        # --- FIX: Added clip=True to marks ---
        scenes_layer = alt.Chart(sub_scenes).mark_line(
            opacity=0.15, strokeWidth=1, clip=True # Clips spaghetti lines
        ).encode(
            x=alt.X('value:Q', title=x_axis_title),
            y=alt.Y(f'{score_col}:Q', scale=y_scale, title=y_title if res_raw == '100%' else ""),
            detail='scene:N', color=alt.value(res_color)
        )
        
        mean_layer = alt.Chart(sub_mean).mark_line(
            strokeWidth=3, point=True, clip=True # Clips mean line
        ).encode(
            x='value:Q', y='mean:Q', color=alt.value(res_color)
        )

        # Rules (also clipped to frame)
        rules = []
        if ue_default_val is not None:
            rules.append(alt.Chart(pd.DataFrame({'x': [ue_default_val]})).mark_rule(
                color='#FFD700', strokeDash=[4, 4], size=1.5, clip=True
            ).encode(x='x:Q'))
        if opt_val is not None:
            rules.append(alt.Chart(pd.DataFrame({'x': [opt_val]})).mark_rule(
                color=res_color, strokeDash=[2, 2], size=1.5, clip=True
            ).encode(x='x:Q'))

        # Delta Calculation
        scene_list = sub_scenes['scene'].unique()
        deltas = []
        for s in scene_list:
            s_data = sub_scenes[sub_scenes['scene'] == s]
            if len(s_data) < 2: continue
            idx_opt = (s_data['value'] - opt_val).abs().idxmin()
            idx_ue  = (s_data['value'] - ue_default_val).abs().idxmin()
            deltas.append(s_data.loc[idx_opt, score_col] - s_data.loc[idx_ue, score_col])
        
        min_d, max_d = min(deltas), max(deltas)
        summary_stats[res_label] = {'min_delta': min_d, 'max_delta': max_d}

        # Multi-line Annotations (Not clipped so they stay visible on the right)
        anno_df = pd.DataFrame([{
            'x': sub_mean['value'].max(), 
            'y': sub_mean['mean'].iloc[-1], 
            'line1': f"max Δ: +{max_d:.2f}",
            'line2': f"min Δ: {'+' if min_d >= 0 else ''}{min_d:.2f}"
        }])
        
        text_max = alt.Chart(anno_df).mark_text(
            align='left', dx=10, dy=-7, fontSize=11, fontWeight='bold', color=res_color
        ).encode(x='x:Q', y='y:Q', text='line1:N')
        
        text_min = alt.Chart(anno_df).mark_text(
            align='left', dx=10, dy=7, fontSize=11, fontWeight='bold', color=res_color
        ).encode(x='x:Q', y='y:Q', text='line2:N')

        charts.append(alt.layer(scenes_layer, mean_layer, *rules, text_max, text_min).properties(
            width=230, height=200))

    final_chart = alt.hconcat(*charts).properties(title=plot_title).configure_axis(
        labelFontSize=12, titleFontSize=13, grid=True
    ).configure_title(fontSize=16, anchor='middle').configure_view(stroke=None)

    return final_chart, summary_stats

In [23]:
# Call for Alpha Weight
alpha_chart, alpha_stats = plot_param_per_scene(
    df_cent, 'alpha_weight', 'score_centered', 'Centered CGVQM', 'Alpha Weight'
)

# Call for History Weight
hist_chart, hist_stats = plot_param_per_scene(
    df_cent, 'hist_percent', 'score_centered', 'Centered CGVQM', 'History Percentage'
)

alpha_chart.save('alpha-weight-allcurves.png', scale_factor=2.0)
hist_chart.save('hist-percent-allcurves.png', scale_factor=2.0)

In [100]:
UNREAL_DEFAULTS = {
    'alpha_weight': 0.04,
    'hist_percent': 100,
    'filter_size':  1,
    'num_samples':  8,
}

def compute_recommendations(df, resolution_labels=RESOLUTION_LABELS):
    recommendations = {}
    results = {}
    
    for param, default_val in UNREAL_DEFAULTS.items():
        param_df = df[df['parameter'] == param].copy()
        mean_scores = (
            param_df.groupby(['resolution', 'value'])['score']
            .mean()
            .reset_index()
        )
        # Optimal (argmax) per resolution
        optimal = mean_scores.loc[mean_scores.groupby('resolution')['score'].idxmax()].copy()
        optimal = optimal.rename(columns={'value': 'optimal_value', 'score': 'optimal_score'})
        # Baseline score at Unreal default
        baseline = (
            mean_scores[mean_scores['value'] == default_val]
            .rename(columns={'score': 'baseline_score'})
            [['resolution', 'baseline_score']]
        )
        # Join and compute gain
        result = optimal.merge(baseline, on='resolution')
        result['gain'] = result['optimal_score'] - result['baseline_score']
        result['resolution'] = result['resolution'].map(resolution_labels)
        result = result.sort_values('resolution', ascending=False)
        
        results[param] = result[['resolution', 'baseline_score', 'optimal_value', 'optimal_score', 'gain']].round(2)
        
        # {resolution_label: optimal_value} for the plot
        recommendations[param] = dict(zip(result['resolution'], result['optimal_value']))
    
    return recommendations, results

RECOMMENDATIONS, rec_results = compute_recommendations(df_raw)

# Display per-parameter tables
for param, result in rec_results.items():
    print(f"\n── {PARAM_LABELS[param]} ──")
    display(result)

print(RECOMMENDATIONS)


── Alpha weight ──


,resolution,baseline_score,optimal_value,optimal_score,gain
2,75%,92.39,0.5,94.28,1.89
1,50%,91.14,0.2,92.64,1.50
0,25%,87.35,0.2,88.60,1.25
3,100%,93.24,0.6,95.26,2.02



── History % ──


,resolution,baseline_score,optimal_value,optimal_score,gain
2,75%,92.41,200.0,93.82,1.41
1,50%,91.13,200.0,92.35,1.22
0,25%,87.35,150.0,87.74,0.38
3,100%,93.34,200.0,94.70,1.36



── Filter size ──


,resolution,baseline_score,optimal_value,optimal_score,gain
2,75%,92.40,0.95,92.41,0.02
1,50%,91.13,0.95,91.14,0.01
0,25%,87.38,1.00,87.38,0.00
3,100%,93.33,0.90,93.34,0.01



── Num samples ──


,resolution,baseline_score,optimal_value,optimal_score,gain
2,75%,92.40,64.0,92.43,0.04
1,50%,91.13,64.0,91.16,0.03
0,25%,87.35,8.0,87.35,0.00
3,100%,93.33,8.0,93.33,0.00


{'alpha_weight': {'75%': 0.5, '50%': 0.2, '25%': 0.2, '100%': 0.6}, 'hist_percent': {'75%': 200.0, '50%': 200.0, '25%': 150.0, '100%': 200.0}, 'filter_size': {'75%': 0.95, '50%': 0.95, '25%': 1.0, '100%': 0.9}, 'num_samples': {'75%': 64.0, '50%': 64.0, '25%': 8.0, '100%': 8.0}}


In [131]:
# ── At module level (outside function) ────────────────────────────────────────
RESOLUTION_LABELS = {100: '100%', 87: '75%', 71: '50%', 50: '25%'}
res_order  = [RESOLUTION_LABELS[k] for k in [100, 87, 71, 50]]  # ['100%', '75%', '50%', '25%']
res_colors = ['#2a7db5', '#2ca05a', '#e07b3d', '#d4504a']
color_scale = alt.Scale(domain=res_order, range=res_colors)


def plot_response_curves(input_df, score_col, y_title, plot_title, y_min=None, y_max=None):
    agg = (
        input_df.groupby(['parameter', 'resolution', 'value'])[score_col]
        .agg(['mean', 'std', 'count'])
        .reset_index()
    )
    agg['ci95'] = 1.96 * (agg['std'] / np.sqrt(agg['count']))
    agg['resolution'] = agg['resolution'].astype(int).map(RESOLUTION_LABELS)
    agg['ci_low']  = agg['mean'] - agg['std']
    agg['ci_high'] = agg['mean'] + agg['std']
    _y_min = y_min if y_min is not None else agg['ci_low'].min() - 0.5
    _y_max = y_max if y_max is not None else agg['ci_high'].max() + 0.5
    y_scale = alt.Scale(domain=[_y_min, _y_max])

    PARAMS_ORDERED = ['alpha_weight', 'hist_percent', 'filter_size', 'num_samples']
    charts = []
    for param in PARAMS_ORDERED:
        sub = agg[agg['parameter'] == param].copy()

        band = (
            alt.Chart(sub)
            .mark_area(opacity=0.2)
            .encode(
                x=alt.X('value:Q', title=PARAM_LABELS[param]),
                y=alt.Y('ci_low:Q', title='', scale=y_scale),
                y2=alt.Y2('ci_high:Q'),
                color=alt.Color('resolution:N', scale=color_scale, sort=res_order, legend=None),
            )
        )

        line = (
            alt.Chart(sub)
            .mark_line(point=True)
            .encode(
                x=alt.X('value:Q', title=PARAM_LABELS[param]),
                y=alt.Y('mean:Q', title=y_title, scale=y_scale),
                color=alt.Color('resolution:N', title='Base Resolution',
                                scale=color_scale, sort=res_order),
                tooltip=[
                    'resolution', 'value',
                    alt.Tooltip('mean:Q', format='.3f'),
                    alt.Tooltip('std:Q',  format='.3f'),
                ],
            )
        )

        # ── Unreal default: single yellow dashed vertical line ─────────────
        default_df = pd.DataFrame([{'x': UNREAL_DEFAULTS[param], 'label': "Unreal default"}])
        default_rule = (
            alt.Chart(default_df)
            .mark_rule(color='gold', strokeDash=[4, 4], strokeWidth=2)
            .encode(
                x=alt.X('x:Q', title=None),
                tooltip=alt.Tooltip('label:N'),
            )
        )

        # ── Per-resolution recommendations: one dashed line per resolution ─
        rec_df = pd.DataFrame([
            {'x': v, 'resolution': res}
            for res, v in RECOMMENDATIONS[param].items()
        ])
        rec_rule = (
            alt.Chart(rec_df)
            .mark_rule(strokeDash=[4, 4], strokeWidth=2)
            .encode(
                x=alt.X('x:Q', title=None),
                color=alt.Color('resolution:N', scale=color_scale, sort=res_order, legend=None),
                tooltip=['resolution', alt.Tooltip('x:Q', title='Recommended value')],
            )
        )

        charts.append(
            (band + line + rec_rule + default_rule)  # default_rule last = on top
            .properties(title=PARAM_LABELS[param], width=250, height=220)
        )

    legend_chart = (
        alt.Chart(pd.DataFrame({'resolution': res_order}))
        .mark_point(size=0, opacity=0)
        .encode(
            color=alt.Color(
                'resolution:N',
                scale=color_scale,
                sort=res_order,
                legend=alt.Legend(title='Base Resolution')
            )
        )
    )

    
    return (
        alt.hconcat(*charts, legend_chart)
        .resolve_scale(color='shared')
        .properties(title=plot_title)
    )

In [132]:
raw_chart = plot_response_curves(
    df_raw,
    score_col='score',
    y_title='Mean CGVQM Score',
    plot_title='TAA Parameter Response Curves (Raw CGVQM)',
    y_min=raw_y_min,
    y_max=raw_y_max
)

cent_chart = plot_response_curves(
    df_cent,
    score_col='score_centered',
    y_title='Mean Centered CGVQM',
    plot_title='TAA Parameter Response Curves (Mean-Centered CGVQM)',
    y_min=cent_y_min,
    y_max=cent_y_max
)

alt.vconcat(raw_chart, cent_chart).resolve_scale(color='shared')

alt.VConcatChart(...)

In [133]:
df_cent = load_df(JSON_PATH, center=True)
agg = df_cent.groupby(['parameter', 'resolution', 'value'])['score_centered'].agg(['mean', 'std', 'count']).reset_index()
agg['ci95'] = 1.96 * (agg['std'] / np.sqrt(agg['count']))
agg['resolution'] = agg['resolution'].astype(str) + '%'

y_min = (agg['mean'] - agg['ci95']).min() - 0.5
y_max = (agg['mean'] + agg['ci95']).max() + 0.5
y_scale = alt.Scale(domain=[y_min, y_max])

charts = []
for param in PARAMS:
    sub = agg[agg['parameter'] == param]
    band = alt.Chart(sub).mark_errorband(opacity=0.2).encode(
        x=alt.X('value:Q', title=param),
        y=alt.Y('mean:Q', scale=y_scale),
        yError='ci95:Q',
        color=alt.Color('resolution:N', scale=color_scale, sort=res_order)
    )
    line = alt.Chart(sub).mark_line(point=True).encode(
        x=alt.X('value:Q', title=param),
        y=alt.Y('mean:Q', title='Mean Centered CGVQM', scale=y_scale),
        color=alt.Color('resolution:N', title='Base Resolution', scale=color_scale, sort=res_order),
        tooltip=['resolution', 'value', alt.Tooltip('mean:Q', format='.3f'), alt.Tooltip('ci95:Q', format='.3f')]
    )
    charts.append((band + line).properties(title=f'{param}', width=280, height=200))

alt.hconcat(*[alt.hconcat(*charts[:2]), alt.hconcat(*charts[2:])]).resolve_scale(color='shared').properties(
    title='TAA Parameter Response Curves (Mean-Centered CGVQM)'
)

/Users/mariasilva/Documents/PerceptualTAA/parameter_analysis/../dataloader.py:103: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(_center_group)


alt.HConcatChart(...)

In [6]:
import json
import altair as alt
import pandas as pd

with open('../dataset.json', 'r') as f:
    raw = json.load(f)

scene = 'wildwest-bar'
res   = '50'
param = 'filter_size'

records = []
for entry in raw[scene][res][f'ref-{scene}'][param]:
    for frame_idx, error in enumerate(entry['per_frame_errors']):
        records.append({
            'frame': frame_idx,
            'error': error,
            'value': entry['value'],
        })

df_frames = pd.DataFrame(records)

# Exclude frames within 3 of a multiple of 30
df_frames = df_frames[df_frames['frame'].apply(lambda f: (f % 30) > 5 and (f % 30) < 25)]
# df_frames = df_frames[(df_frames['frame'] > 10) & (df_frames['frame'] < 140)]

alt.Chart(df_frames).mark_line(opacity=0.6, strokeWidth=1).encode(
    x=alt.X('frame:Q', title='Frame'),
    y=alt.Y('error:Q', title='Per-frame error (lower = better)'),
    color=alt.Color('value:O', title='alpha_weight value'),
    tooltip=['frame', 'error', 'value']
).properties(
    title=f'{scene} — {param} — resolution {res}% (boundary frames excluded)',
    width=500,
    height=300
)

alt.Chart(...)